<a href="https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")
EXTRACT_PATH = Path("/content/flyrank_project")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ ZIP extracted!")
print("CSV files found:")

for p in EXTRACT_PATH.rglob("content_refresh_anonymized.csv"):
    print(p)

✅ ZIP extracted!
CSV files found:
/content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv


## 1. Question

### Research Question

Can observable content and search-performance signals identify pages that are more likely to be declining and therefore worth review?

### Decision Supported

The goal is to help a content strategist or SEO reviewer **prioritise which pages should be reviewed first**. The model provides a ranking signal that can help create a focused review queue.

This is **directional decision-support**, not a claim that the model can prove which pages will decline or that refreshing a page will cause better performance.

In [1]:
import pandas as pd

question_summary = pd.DataFrame([
    {
        "Research question":
            "Can observable content and search-performance signals identify pages "
            "that are more likely to be declining and therefore worth review?",
        "Decision supported":
            "Prioritise pages for human content review",
        "Intended use":
            "Directional decision-support",
        "Not established":
            "The model does not prove future decline or that refreshing a page causes improvement"
    }
])

display(question_summary)

,Research question,Decision supported,Intended use,Not established
0,Can observable content and search-performance ...,Prioritise pages for human content review,Directional decision-support,The model does not prove future decline or tha...


## 2. Data

### Dataset

This analysis uses the FlyRank ML Internship content-refresh dataset. The dataset contains anonymized content and search-performance observations used to study content decline and prioritise pages for review.

### Data Used

The analysis uses the following observable signals:

- Content age
- Days since last update
- 90-day impressions
- Average search position
- Click-through rate (CTR)
- Word count

The target label is derived from `trend_direction`:

- `down` → declining page
- Other trend categories → not labelled as declining

### Time Context

The dataset uses search-performance signals measured over a 90-day window. Trend direction is based on the comparison of the most recent 30 days with the preceding 30 days.

### Exclusions

The following fields were excluded from model features:

- `trend_direction` — directly defines the target
- `trend_pct` — directly encodes the outcome
- `content_id` — identifier only
- `client_id` — used only for grouped validation, not as a predictive feature

These exclusions reduce the risk of target leakage and keep the model focused on observable content and performance signals.

### Public-Safety

The analysis uses anonymized/public-safe data only. No client names, private queries, URLs, or other identifying information are included in the paper.

In [4]:
from pathlib import Path
import zipfile
import pandas as pd

# Find the extracted FlyRank project
possible_paths = [
    Path("/content/flyrank_project"),
    Path("/content/flyrank-ml-internship-main"),
]

DATA_PATH = None

for base in possible_paths:
    if base.exists():
        matches = list(base.rglob("content_refresh_anonymized.csv"))
        if matches:
            DATA_PATH = matches[0]
            break

if DATA_PATH is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found.")

print("Dataset:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Dataset: /content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Methodology

### Target

The target is `is_declining_label`, where:

- `1` = `trend_direction == "down"`
- `0` = not declining

### Features

The model uses observable content and search-performance signals:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`
- `word_count`

### Models

Three supervised classification approaches were evaluated:

1. Logistic Regression — simple linear model and interpretable reference.
2. Decision Tree — provides an easier-to-interpret rule-based model.
3. Random Forest — captures non-linear relationships and interactions.

### Baseline

The Week-4 rule-based baseline is also evaluated:

`impressions_90d >= 500 AND avg_position > 0 AND ctr < 0.5`

### Validation

The preferred validation design is a client-holdout split. Twenty percent of clients are held out for testing so that pages from the same client do not appear in both training and test data.

The baseline and all models are evaluated on the **same held-out rows** using **Precision@50** as the primary ranking metric.

### Leakage Check

`trend_direction` and `trend_pct` are excluded from model features because they define or directly encode the target.

`content_id` and `client_id` are also not used as predictive features. `client_id` is used only to create the grouped validation split.

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------
# 1. Prepare target
# -----------------------------
df = df.copy()

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# -----------------------------
# 2. Select model features
# -----------------------------
FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[FEATURES].copy()
y = df["is_declining_label"].copy()

print("Features used:")
print(FEATURES)
print("\nTarget distribution:")
print(y.value_counts())

# -----------------------------
# 3. Client-holdout split
# -----------------------------
clients = df["client_id"].dropna().unique()

rng = np.random.RandomState(42)
rng.shuffle(clients)

n_test_clients = max(1, int(len(clients) * 0.20))
test_clients = set(clients[:n_test_clients])

test_mask = df["client_id"].isin(test_clients)

X_train = X.loc[~test_mask]
X_test = X.loc[test_mask]
y_train = y.loc[~test_mask]
y_test = y.loc[test_mask]

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", df.loc[~test_mask, "client_id"].nunique())
print("Test clients:", df.loc[test_mask, "client_id"].nunique())

# Confirm no client overlap
overlap = set(
    df.loc[~test_mask, "client_id"]
) & set(
    df.loc[test_mask, "client_id"]
)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

# -----------------------------
# 4. Precision@50
# -----------------------------
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]

    return float(y_true[order].mean())

# -----------------------------
# 5. Models
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        ))
    ])
}

# -----------------------------
# 6. Week-4 baseline
# -----------------------------
baseline_scores = (
    (
        (X_test["impressions_90d"] >= 500)
        & (X_test["avg_position"] > 0)
        & (X_test["ctr"] < 0.5)
    )
    .astype(float)
    .values
)

results = [
    {
        "Method": "Week-4 baseline",
        "Precision@50": precision_at_k(
            y_test.values, baseline_scores, 50
        )
    }
]

# -----------------------------
# 7. Train + evaluate models
# -----------------------------
model_scores = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    scores = model.predict_proba(X_test)[:, 1]
    model_scores[name] = scores

    predictions = (scores >= 0.5).astype(int)

    results.append({
        "Method": name,
        "Precision@50": precision_at_k(
            y_test.values, scores, 50
        ),
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Average Precision": average_precision_score(y_test, scores),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        )
    })

comparison = (
    pd.DataFrame(results)
    .sort_values("Precision@50", ascending=False)
    .reset_index(drop=True)
)

display(comparison.round(3))

# -----------------------------
# 8. Leakage audit
# -----------------------------
FORBIDDEN = {
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

assert not (set(FEATURES) & FORBIDDEN)

print("Leakage audit: PASSED")
print("No target-derived fields or identifiers are used as model features.")

# -----------------------------
# 9. Best model
# -----------------------------
best_model_row = comparison[
    comparison["Method"] != "Week-4 baseline"
].iloc[0]

best_model_name = best_model_row["Method"]

baseline_p50 = comparison.loc[
    comparison["Method"] == "Week-4 baseline",
    "Precision@50"
].iloc[0]

best_p50 = best_model_row["Precision@50"]

print("\nBest ML model:", best_model_name)
print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Best-model Precision@50: {best_p50:.3f}")
print(f"Difference: {best_p50 - baseline_p50:+.3f}")

Features used:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Train rows: 26619
Test rows: 3381
Train clients: 26
Test clients: 6
Client overlap: 0


,Method,Precision@50,ROC-AUC,Average Precision,Precision,Recall,F1
0,Week-4 baseline,0.58,NaN,NaN,NaN,NaN,NaN
1,Random Forest,0.54,0.655,0.627,0.608,0.759,0.675
2,Logistic Regression,0.44,0.551,0.550,0.556,0.825,0.664
3,Decision Tree,0.36,0.641,0.610,0.622,0.630,0.626


Leakage audit: PASSED
No target-derived fields or identifiers are used as model features.

Best ML model: Random Forest
Baseline Precision@50: 0.580
Best-model Precision@50: 0.540
Difference: -0.040


## 4. Results (vs baseline)

The baseline and all machine-learning models were evaluated on the same client-holdout test set.

### Primary Metric: Precision@50

Precision@50 is the primary metric because the practical use case is to create a small ranked review queue.

It answers:

> Among the 50 highest-ranked pages, what fraction are actually declining?

### Comparison

The results below compare the Week-4 rule-based baseline with Logistic Regression, Decision Tree, and Random Forest.

The model with the strongest Precision@50 is treated as the preferred ranking model for the action playbook.

The result should be interpreted as a measured ranking performance on this test split, not as proof of future decline or causal impact of refreshing content.

In [6]:
# Display the model vs baseline results
results_table = comparison.copy()

display(results_table.round(3))

# Highlight the primary comparison
best_model_row = results_table[
    results_table["Method"] != "Week-4 baseline"
].iloc[0]

baseline_row = results_table[
    results_table["Method"] == "Week-4 baseline"
].iloc[0]

print("Primary result")
print("----------------")
print(
    f"Best ML model: {best_model_row['Method']}"
)
print(
    f"Best-model Precision@50: "
    f"{best_model_row['Precision@50']:.3f}"
)
print(
    f"Week-4 baseline Precision@50: "
    f"{baseline_row['Precision@50']:.3f}"
)
print(
    f"Difference: "
    f"{best_model_row['Precision@50'] - baseline_row['Precision@50']:+.3f}"
)

,Method,Precision@50,ROC-AUC,Average Precision,Precision,Recall,F1
0,Week-4 baseline,0.58,NaN,NaN,NaN,NaN,NaN
1,Random Forest,0.54,0.655,0.627,0.608,0.759,0.675
2,Logistic Regression,0.44,0.551,0.550,0.556,0.825,0.664
3,Decision Tree,0.36,0.641,0.610,0.622,0.630,0.626


Primary result
----------------
Best ML model: Random Forest
Best-model Precision@50: 0.540
Week-4 baseline Precision@50: 0.580
Difference: -0.040


## 4. Results (vs baseline)

The baseline and all machine-learning models were evaluated on the same client-holdout test set.

### Primary Metric: Precision@50

Precision@50 is the primary metric because the practical use case is to create a small ranked review queue.

Among the evaluated machine-learning models, Random Forest achieved the highest Precision@50 at **0.54**.

However, the Week-4 rule-based baseline achieved a higher Precision@50 of **0.58** on the same test set.

Therefore, the machine-learning models did **not outperform the existing baseline** under this validation design.

Random Forest was the strongest of the tested ML models, but the result does not justify replacing the simpler baseline. This is treated as a measured result on one held-out split rather than evidence that a more complex model is inherently better.

The additional metrics provide context on model behaviour beyond the top-50 ranking performance.


In [7]:
# Simple interpretation of the primary result

baseline_p50 = 0.58
rf_p50 = 0.54

if rf_p50 > baseline_p50:
    conclusion = "Random Forest outperformed the Week-4 baseline."
elif rf_p50 < baseline_p50:
    conclusion = "Random Forest did not outperform the Week-4 baseline."
else:
    conclusion = "Random Forest matched the Week-4 baseline."

print(conclusion)
print(f"Baseline Precision@50: {baseline_p50:.2f}")
print(f"Random Forest Precision@50: {rf_p50:.2f}")
print(f"Difference: {rf_p50 - baseline_p50:+.2f}")

Random Forest did not outperform the Week-4 baseline.
Baseline Precision@50: 0.58
Random Forest Precision@50: 0.54
Difference: -0.04


## 5. Limitations

The results should be interpreted within the scope of this dataset and validation design.

### Model and Data Limitations

- The dataset is an anonymized 30,000-row internship dataset and does not represent the full FlyRank research dataset.
- The target label is derived from observed `trend_direction`; it describes measured decline rather than proving future decline.
- The model uses a limited set of observable content and search-performance features. Important factors such as search intent changes, seasonality, technical SEO issues, competition changes, and external events are not included.
- The client-holdout split provides a more conservative evaluation, but it is still based on one split and should not be treated as universal performance.
- Random Forest achieved the strongest ML performance, but its Precision@50 of 0.54 was below the Week-4 baseline of 0.58.
- Therefore, the results do not justify claiming that the ML model is better than the simpler rule-based approach.

### Operational Limitations

The model should be used as a **decision-support and prioritisation tool**, not as an automated content-management system.

A high model score does not by itself justify publishing, deleting, rewriting, redirecting, or refreshing a page. Human review is required before taking content actions.

The model also does not establish that refreshing a page will cause improved performance.

In [8]:
# Summarise the main limitations as a reproducibility check

limitations = pd.DataFrame([
    {
        "Area": "Dataset",
        "Limitation": "Anonymized 30,000-row internship dataset; not the full research dataset"
    },
    {
        "Area": "Target",
        "Limitation": "Decline label reflects observed trend_direction, not proven future decline"
    },
    {
        "Area": "Features",
        "Limitation": "Does not capture seasonality, intent changes, technical SEO, competition, or external events"
    },
    {
        "Area": "Validation",
        "Limitation": "Client-holdout evaluation is based on one held-out split"
    },
    {
        "Area": "Model performance",
        "Limitation": "Random Forest Precision@50 (0.54) was below the Week-4 baseline (0.58)"
    },
    {
        "Area": "Use",
        "Limitation": "Output is decision-support; human review is required before content actions"
    }
])

display(limitations)

,Area,Limitation
0,Dataset,"Anonymized 30,000-row internship dataset; not ..."
1,Target,Decline label reflects observed trend_directio...
2,Features,"Does not capture seasonality, intent changes, ..."
3,Validation,Client-holdout evaluation is based on one held...
4,Model performance,Random Forest Precision@50 (0.54) was below th...
5,Use,Output is decision-support; human review is re...


## 6. Ranked Recommendations

The model output is intended to help content and SEO reviewers prioritise pages for human review.

### Recommended Action Playbook

| Priority Signal | Reason Code | Recommended Action |
|---|---|---|
| Low CTR + stale content | `LOW_CTR_AND_STALE` | Review title, meta description, search intent, and content freshness |
| Low CTR | `LOW_CTR` | Review title, snippet alignment, and search intent |
| Stale content | `STALE_CONTENT` | Review whether information is outdated and whether a refresh is justified |
| No strong signal | `MONITOR` | Continue monitoring before taking action |

### Human Review

Before taking action, the reviewer should check:

1. Whether the page still matches the intended search intent.
2. Whether the information is outdated or inaccurate.
3. Whether CTR is weak because of the search result snippet or another factor.
4. Whether there are seasonal, competitive, or technical reasons for the observed performance.
5. Whether the proposed action is appropriate for the page and business context.

### What Should NOT Be Automated

The model should not automatically:

- Publish or delete content.
- Change canonical URLs, redirects, or indexing settings.
- Make medical, legal, financial, or brand-safety decisions.
- Rewrite content solely because of a high model score.
- Treat a refresh recommendation as evidence of causal improvement.

The ranked output is therefore a **prioritisation aid for human review**, rather than an autonomous content decision system.

In [9]:
# Create a transparent action-playbook table

action_playbook = pd.DataFrame([
    {
        "Priority signal": "Low CTR + stale content",
        "Reason code": "LOW_CTR_AND_STALE",
        "Recommended action": "Review CTR, search intent, and content freshness"
    },
    {
        "Priority signal": "Low CTR",
        "Reason code": "LOW_CTR",
        "Recommended action": "Review title, snippet alignment, and search intent"
    },
    {
        "Priority signal": "Stale content",
        "Reason code": "STALE_CONTENT",
        "Recommended action": "Review whether content is outdated and needs refresh"
    },
    {
        "Priority signal": "No strong signal",
        "Reason code": "MONITOR",
        "Recommended action": "Continue monitoring before taking action"
    }
])

display(action_playbook)

,Priority signal,Reason code,Recommended action
0,Low CTR + stale content,LOW_CTR_AND_STALE,"Review CTR, search intent, and content freshness"
1,Low CTR,LOW_CTR,"Review title, snippet alignment, and search in..."
2,Stale content,STALE_CONTENT,Review whether content is outdated and needs r...
3,No strong signal,MONITOR,Continue monitoring before taking action


## 7. Artifacts the Paper Embeds

The research paper is supported by reproducible project artifacts generated during the internship.

### Included Artifacts

- Week-4 rule-based baseline and ranked review queue
- Week-5 model comparison and evaluation
- Week-6 validation and leakage audit
- Week-7 action playbook and ranked recommendations
- Week-8 capstone notebook

### Reproducibility

The notebooks document the progression from baseline development to model evaluation, validation auditing, and operational recommendations.

The project repository provides the implementation and notebook history needed to reproduce the analysis.

In [10]:
artifacts = pd.DataFrame([
    {
        "Artifact": "Week 4",
        "Purpose": "Rule-based baseline and ranked review queue"
    },
    {
        "Artifact": "Week 5",
        "Purpose": "Model training and comparison against baseline"
    },
    {
        "Artifact": "Week 6",
        "Purpose": "Validation audit and leakage checks"
    },
    {
        "Artifact": "Week 7",
        "Purpose": "Ranked actions and content action playbook"
    },
    {
        "Artifact": "Week 8",
        "Purpose": "Capstone research paper assembly"
    }
])

display(artifacts)

,Artifact,Purpose
0,Week 4,Rule-based baseline and ranked review queue
1,Week 5,Model training and comparison against baseline
2,Week 6,Validation audit and leakage checks
3,Week 7,Ranked actions and content action playbook
4,Week 8,Capstone research paper assembly


In [11]:
print("CAPSTONE SELF-CHECK")
print("=" * 40)

checks = {
    "Question documented": True,
    "Data documented": True,
    "Methodology documented": True,
    "Results vs baseline documented": True,
    "Limitations documented": True,
    "Ranked recommendations documented": True,
    "Artifacts documented": True,
    "Leakage risk addressed": True,
    "Human review required": True,
    "Causal claims avoided": True,
}

for item, status in checks.items():
    print(f"{'✅' if status else '❌'} {item}")

assert all(checks.values())

print("\n✅ All capstone notebook checks passed.")

CAPSTONE SELF-CHECK
✅ Question documented
✅ Data documented
✅ Methodology documented
✅ Results vs baseline documented
✅ Limitations documented
✅ Ranked recommendations documented
✅ Artifacts documented
✅ Leakage risk addressed
✅ Human review required
✅ Causal claims avoided

✅ All capstone notebook checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
